# Canary-Qwen Batch Evaluation

This notebook evaluates the NVIDIA Canary-Qwen-2.5B model on a dataset of audio segments.
It uses the NeMo framework directly for batch inference.

In [ ]:
#@title Install bleeding-edge packages for Canary-Qwen
# NOTE: This will upgrade the shared environment on the VM!
!pip install torch torchvision torchaudio
!pip install peft transformers huggingface_hub
!pip install "nemo_toolkit[asr,tts] @ git+https://github.com/NVIDIA/NeMo.git"

# After running this cell, you MUST restart the kernel (Kernel -> Restart)!

In [ ]:
import os
import json
import torch
from google.cloud import storage
from nemo.collections.speechlm2.models import SALM

# Import common GCS utils and runner (CORRECTED)
from common.gcs_utils import parse_gcs_uri, download_blob_to_file, download_jsonl_manifest, upload_inference_results
from common.eval_runner import run_batch_evaluation

# Configure logging
import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

print("Loading NVIDIA Canary-Qwen-2.5B into GPU VRAM...")
model = SALM.from_pretrained('nvidia/canary-qwen-2.5b').half().eval().to("cuda")

In [ ]:
# Configuration
GCP_PROJECT_ID = "<YOUR_GCP_PROJECT_ID>"
GCS_MANIFEST_URI = "<YOUR_GCS_MANIFEST_URI>"
GCS_BUCKET = "<YOUR_GCS_BUCKET_NAME>" 
PROJECT_NAME = "<YOUR_PROJECT_NAME>"
EXPERIMENT_NAME = "<YOUR_EXPERIMENT_NAME>"
MODEL_NAME = "<MODEL_NAME>"
BATCH_SIZE = 4  # Smaller batch size for safety
LIMIT = 10  # For testing

In [ ]:
#@title Define helper functions and Run Evaluation

def prompt_formatter(entry, local_path):
    """Define prompt formatter for Canary-Qwen."""
    dialog = [
        {
            "role": "user",
            # 1. You must include the audio locator tag in the text
            "content": "Transcribe the following: <|audioplaceholder|>",
            # 2. Pass the file path directly into the dictionary
            "audio": [local_path] 
        }
    ]
    # Returning as a tuple assuming your batch runner expects it
    return (dialog, local_path)

def canary_inference(model, prompts):
    """Runs inference using the Canary-Qwen model natively."""
    # Unpack just the dialog dictionaries
    dialogs = [p[0] for p in prompts]
    
    # The model's generate method handles all audio loading and padding natively
    answer_ids = model.generate(
        prompts=dialogs, 
        max_new_tokens=256
    )
    
    return answer_ids

def result_decoder(ans, model):
    """Extracts transcription from model output."""
    # Ensure tensor is on CPU before converting to list
    token_ids = ans.cpu().tolist()
    text = model.tokenizer.ids_to_text(token_ids)
    
    # Clean up standard LLM chat template artifacts if they bleed through
    if "transcript" in text:
        text = text.split("transcript")[-1]
    elif "<|imstart|>assistant" in text:
        text = text.split("<|imstart|>assistant")[-1]
        
    return text.strip()

storage_client = storage.Client(project=GCP_PROJECT_ID)
manifest_data = download_jsonl_manifest(storage_client, GCS_MANIFEST_URI)

# Run the generic batch evaluation
results_list = run_batch_evaluation(
    model=model,
    manifest_data=manifest_data,
    prompt_fn=prompt_formatter,
    inference_fn=canary_inference,
    decode_fn=result_decoder,
    storage_client=storage_client,
    project_name=PROJECT_NAME,
    selected_model=MODEL_NAME,
    batch_size=BATCH_SIZE,
    limit=LIMIT
)

In [ ]:
# Upload results directly to GCS from memory (uncomment to use)
gcs_uri = upload_inference_results(
    storage_client, 
    GCS_BUCKET, 
    PROJECT_NAME, 
    MODEL_NAME, 
    EXPERIMENT_NAME, 
    results_list
)